# Group 5 v3 Model
Chest X-ray Multiclassifier
Uses Asymmetric Loss known for greater AUC for X-rays in particular.
I attempted to use Mixup but it can distort chest x-ray structures so it has been removed.
Therefore, TODO, rename file when not running
Swin_s is used in this model, not Swin_t which has half the parameters.


### Changes I made to get it to (hopefully) work

remove HorizontalFlip, not useful in chest x-rays
decrease RandomRotation, 0-3 degrees max
add PA/AP data


### v4 brainstorming while this runs:

add SSL

normalize orientation (other than PA/AP)


In [3]:
import numpy as np
from PIL import Image
from tqdm import tqdm
import os
import glob
# Build image-path index
IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"
# Recursively find every PNG once and build a filename -> full-path dict
all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
path_lookup = {os.path.basename(p): p for p in all_png}
print(f"Found {len(path_lookup):,} images on disk.")



import sys
import subprocess

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# required for tqdm notebook progress bars
install_if_missing("ipywidgets")
install_if_missing("tqdm")

Found 112,120 images on disk.


In [ ]:
"""
Author: Nicholas J. Calabro

Copyright (c) 2026 Nicholas J. Calabro is licensed under the MIT License.
"""
import glob
import pandas as pd
import numpy as np
import os
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from torch.amp import GradScaler, autocast

import cv2

import torch



### File Parameters (setup options here) ###

# Metadata containing labels and view positions
METADATA_CSV_PATH = "../chest_xray_dataset/CXR8/Data_Entry_2017_v2020.csv"

# Path to directoy containing all CXR8 images (should contain subdirs with PNGs)
IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"


# Pretrained SwinV2 checkpoint (can be downloaded from timm or trained yourself with their codebase)
    # download: https://huggingface.co/zdaxie/SimMIM/blob/main/simmim_swinv2_pretrain_models/swinv2_small_1k_500k.pth
# PRETRAINED_CKPT = "../chest_xray_dataset/swinv2_small_1k_500k.pth"
# NO LONGER NEEDED I am opting for timm's built in pre training because SimMIM has major model differences

# Destination for saving model checkpoints
MODEL_OUTPUT_FILE = "swin_cxr8_best.pth"

### Preprocessing Parameters ###

# CLAHE (contrast limited adaptive histogram equalization parameters)
    # CLIP_LIMIT is the threshold for contrast limiting
    # TILE_GRID_SIZE determines how large a region to compute intesity before adjusting contrast
    # CLAHE_PROB is the probability of applying CLAHE to each image
CLIP_LIMIT = 1.5
TILE_GRID_SIZE = 4
CLAHE_PROB = 0.3

ROTAION_DEGREES = 1.5
TRANSLATION = 0.02

AFFINE_DEGREES = 1.5
AFFINE_TRANSLATION = 0.02
AFFINE_PROB = 0.5


### Model Parameters ###
WARMUP_EPOCHS = 3
WARMUP_START_FACTOR = 0.3
WARMUP_END_FACTOR = 1.0

NUM_EPOCHS = 22
BASE_LR = 4e-5
PATIENCE  = 4

WEIGHT_DECAY = 1e-2
# Don't over sample too much
# Rare amplification compounds SAMPLER_POWER
SAMPLER_POWER = 0.05

GAMMA_NEG = 1
GAMMA_POS = 0

BATCH_SIZE_VAL = 16
BATCH_SIZE_TRAIN = 32


## Dataset Constants ###

NIH_MEAN = [0.49798396, 0.49798396, 0.49798396]
NIH_STD  = [0.24942584, 0.24942584, 0.24942584]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# rare -> least occuring 3 diseases
RARE_CLASSES = ["Hernia", "Fibrosis", "Pneumonia"]
ALL_CLASSES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "No Finding", "Nodule",
    "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]



# Progressive resize schedule: epoch (1-indexed) -> image size
# UPDATE: I am setting it uniformly to 336 x 336, progressive resize isn't effective on Swin
PROG_SCHEDULE = {e: 256 for e in range(1, NUM_EPOCHS + 1)}
# PROG_SCHEDULE = {
#     **{e: 336 for e in range(1, 4)},   # epochs 1-3:  128x128
#     **{e: 336 for e in range(4, 7)},   # epochs 4-6:  192x192
#     **{e: 336 for e in range(7, 11)},  # epochs 7-10: 224x224
# }

def fix_rotation(img):
    w, h = img.size
    if w > h:
        img = img.rotate(90, expand=True)
    return img


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit    = clip_limit        # just store the params
        self.tile_grid_size = tile_grid_size

    def __call__(self, img: Image.Image) -> Image.Image:
        clahe = cv2.createCLAHE(               # fresh object each time, no sharing
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        return Image.fromarray(enhanced)
    

class PerImageStandardize:
    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        # img: (C, H, W)
        mean = img.mean()
        std  = img.std().clamp(min=1e-6)
        return (img - mean) / std

def make_train_tf(size):
    """
    Builds a training transform pipeline for a given resolution.
    CenterCrop is skipped for smaller sizes since the image is already small.
    """
    ops = [
        transforms.Resize((size, size)),
        transforms.RandomApply([
            CLAHETransform(clip_limit=CLIP_LIMIT, tile_grid_size=(TILE_GRID_SIZE, TILE_GRID_SIZE))
        ], p=CLAHE_PROB),
        # CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
        transforms.RandomApply([
            transforms.RandomAffine(AFFINE_DEGREES, translate=(AFFINE_TRANSLATION, AFFINE_TRANSLATION))
        ], p=AFFINE_PROB),
    ]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        # transforms.Normalize(NIH_MEAN, NIH_STD)
    ]

    return transforms.Compose(ops)

def worker_init_fn(worker_id):
    cv2.setNumThreads(0)
        
def check_swin_compat(model, sd):
    model_dim = model.embed_dim

    for k, v in sd.items():
        if "patch_embed.proj.weight" in k:
            ckpt_dim = v.shape[0]
            print(f"Model dim: {model_dim}, CKPT dim: {ckpt_dim}")

            if model_dim != ckpt_dim:
                raise ValueError(
                    f" Architecture mismatch: model={model_dim}, checkpoint={ckpt_dim}"
                )
            return


def extract_state_dict(ckpt):
    if "model" in ckpt:
        return ckpt["model"]
    if "state_dict" in ckpt:
        return ckpt["state_dict"]
    return ckpt

def build_param_groups(model, base_lr=1e-4, decay=0.85):
    layers = [model.backbone.layers[i] for i in range(len(model.backbone.layers))]  # ← fix
    groups = []
    for i, layer in enumerate(reversed(layers)):
        lr = base_lr * (decay ** i)
        groups.append({"params": layer.parameters(), "lr": lr})
    groups.append({"params": model.head.parameters(), "lr": base_lr})
    # also include view_embed and view_proj at head LR
    groups.append({"params": list(model.view_embed.parameters()) + 
                              list(model.view_proj.parameters()), "lr": base_lr})
    registered = set(id(p) for g in groups for p in g["params"])
    others = [p for p in model.parameters() if id(p) not in registered]
    if others:
        groups.append({"params": others, "lr": base_lr * (decay ** len(layers))})
    return groups



# Asymmetric Loss down-weights easy negatives which may be effective for X-ray multi classification
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=GAMMA_NEG, gamma_pos=GAMMA_POS, clip=0.05, eps=1e-6):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        logits = logits * 0.5  # key stability trick

        # probs = torch.sigmoid(logits)
        probs = torch.sigmoid(logits.clamp(-10, 10))

        loss_pos = targets * torch.log(probs.clamp(min=1e-6))
        loss_neg = (1 - targets) * torch.log((1 - probs).clamp(min=1e-6))

        loss = loss_pos * (1 - probs) ** self.gamma_pos + \
            loss_neg * probs ** self.gamma_neg

        return -loss.mean()

class CXR8Dataset(Dataset):
     def __init__(self, df, labels, idx_array, transform, lookup):
         self.df        = df.iloc[idx_array].reset_index(drop=True)
         self.labels    = labels[idx_array]
         self.transform = transform
         self.lookup    = lookup
     def __len__(self):
         return len(self.df)
     def __getitem__(self, i):
         fname = self.df.loc[i, "Image Index"]
         img = Image.open(self.lookup[fname]).convert("L")  # grayscale

         # expand to 3 channels for Swin
         img = np.array(img).astype(np.float32)

         # add VERY small noise
         # img = img + np.random.normal(0, 0.3, img.shape)
         # img = img + np.random.normal(0, 0.1, img.shape)


         # clip to valid range
         img = np.clip(img, 0, 255).astype(np.uint8)

         # expand to 3 channels
         img = np.stack([img, img, img], axis=-1)

         img = Image.fromarray(img)
         img   = fix_rotation(img)
         img   = self.transform(img)
         lbl   = torch.tensor(self.labels[i], dtype=torch.float32)
         view_id  = torch.tensor(self.df.loc[i, "view_id"], dtype=torch.long)
         return img, lbl, view_id
     

class SwinWithView(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()

        C = backbone.head.in_features  # feature dim before original classifier

        # Strip original classifier head
        backbone.head = nn.Identity()
        self.backbone = backbone

        # AP/PA embedding
        self.view_embed = nn.Embedding(2, 32)
        self.view_proj  = nn.Linear(32, C)
        self.view_scale = nn.Parameter(torch.tensor(0.3))

        # New classification head
        self.head = nn.Sequential(
            nn.LayerNorm(C),
            nn.Dropout(0.2),
            nn.Linear(C, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )

    def forward(self, x, view_id):
        feats = self.backbone.forward_features(x)   # (B, H, W, C)

        if feats.ndim == 4:
            feats = feats.mean(dim=(1, 2))          # -> (B, C)

        v = self.view_embed(view_id)
        v = self.view_proj(v)

        feats = feats + self.view_scale * v

        return self.head(feats)





### Constants derived from constant parameters above ###

NO_FINDING_COL = ALL_CLASSES.index("No Finding")
RARE_COLS = [ALL_CLASSES.index(c) for c in RARE_CLASSES]
NUM_CLASSES = len(ALL_CLASSES)

### Main Driver Code ###
if __name__ == "__main__":
    
    # Prepare GPU
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    # import multiprocessing
    # multiprocessing.set_start_method("spawn", force=True)

    # import torch.multiprocessing as mp
    # mp.set_sharing_strategy("file_system")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Enable cuDNN benchmark for potential speedup
    torch.backends.cudnn.benchmark = True

    
    # Train / eval loops
    def run_epoch(loader, train=True, scaler=None):
        model.train() if train else model.eval()
        total_loss = 0.0
        all_logits, all_labels = [], []

        with torch.set_grad_enabled(train):
            for imgs, lbls, views in tqdm(loader, desc="train" if train else "val ", leave=False):
                imgs  = imgs.to(device)
                lbls  = lbls.to(device)
                views = views.to(device)

                if train:
                    original_lbls = lbls.clone()
                    optimizer.zero_grad()

                # with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(scaler is not None)):
                with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
                    logits = model(imgs, views)

                if torch.isnan(logits).any() or not torch.isfinite(logits).all():
                    print("Bad logits - skipping batch")
                    continue

                loss = criterion(logits.float(), lbls.float())

                if torch.isnan(loss) or not torch.isfinite(loss):
                    print("Bad loss - skipping batch")
                    continue

                if train:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)

                    bad_grad = False
                    for name, param in model.named_parameters():
                        if param.grad is not None and torch.isnan(param.grad).any():
                            print(f"NaN gradient in {name} — skipping batch")
                            bad_grad = True
                            break
                    if bad_grad:
                        optimizer.zero_grad()
                        continue

                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()

                total_loss += loss.item() * imgs.size(0)

                all_logits.append(logits.sigmoid().float().cpu().detach())
                all_labels.append((original_lbls if train else lbls).cpu().detach())

        avg_loss = total_loss / len(loader.dataset)

        probs  = torch.cat(all_logits).numpy()
        labels = torch.cat(all_labels).numpy()

        if np.isnan(probs).mean() > 0:
            print("NaNs in predictions — skipping AUC")
            return avg_loss, 0.0, {}

        aucs = []
        per_class_auc = {}

        for c in range(labels.shape[1]):
            col = labels[:, c]
            if col.sum() > 0 and col.sum() < len(col):
                auc = roc_auc_score(labels[:, c], probs[:, c])
                per_class_auc[ALL_CLASSES[c]] = round(auc, 3)
                aucs.append(auc)

        return avg_loss, np.mean(aucs) if aucs else 0.0, per_class_auc

    
    # Load labels
    data_frame = pd.read_csv(METADATA_CSV_PATH)

    # Keep only the two columns we need
    data_frame = data_frame[["Image Index", "Finding Labels", "View Position"]].copy()

    # Convert View Position to integer
    data_frame["view_id"] = data_frame["View Position"].map({"PA": 0, "AP": 1})
    data_frame["view_id"] = data_frame["view_id"].fillna(0).astype(int)

    # Remove Lateral Views
    data_frame = data_frame[data_frame["View Position"].isin(["PA", "AP"])]
    data_frame = data_frame.reset_index(drop=True)

    # Parse multi-label strings  e.g. "Atelectasis|Cardiomegaly"
    data_frame["labels"] = data_frame["Finding Labels"].str.split("|")


    # Recursively find every PNG once and build a filename -> full-path dict
    all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
    path_lookup = {os.path.basename(p): p for p in all_png}
    print(f"Found {len(path_lookup):,} images on disk.")

    mlb = MultiLabelBinarizer(classes=ALL_CLASSES)
    label_matrix = mlb.fit_transform(data_frame["labels"])  # (N, 15)
    rare_idx = np.where(label_matrix[:, RARE_COLS].sum(axis=1) > 0)[0]



    # Filter dataframe to images that actually exist
    mask = data_frame["Image Index"].isin(path_lookup)
    data_frame = data_frame[mask].reset_index(drop=True)
    label_matrix = label_matrix[mask.values]
    print(f"Matched {len(data_frame):,} rows after filtering.")

    # Train / val split
    indices = np.arange(len(data_frame))
    train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)



    # Separate train indices into No-Finding vs everything else
    nf_mask        = label_matrix[train_idx, NO_FINDING_COL] == 1
    nf_train_idx   = train_idx[nf_mask]
    other_train_idx = train_idx[~nf_mask]

    # Keep only 2/3 of No-Finding samples
    keep_n     = int(len(nf_train_idx) * 2 / 3)
    rng        = np.random.default_rng(42)
    nf_kept    = rng.choice(nf_train_idx, size=keep_n, replace=False)

    train_idx  = np.concatenate([other_train_idx, nf_kept])
    print(f"No-Finding kept: {keep_n:,} / {nf_mask.sum():,}  |  New train size: {len(train_idx):,}")

    # train_idx = np.unique(np.concatenate([train_idx, rare_idx]))  # add np.unique
    # train_idx = np.concatenate([train_idx, rare_idx])

    val_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        CLAHETransform(clip_limit=1.5, tile_grid_size=(4,4)),
        transforms.ToTensor(),
        # transforms.Normalize(NIH_MEAN, NIH_STD)
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)  # ← fix this
        # PerImageStandardize(),
    ])



    train_tf = make_train_tf(PROG_SCHEDULE[1])
    train_ds = CXR8Dataset(data_frame, label_matrix, train_idx, train_tf, path_lookup)
    val_ds   = CXR8Dataset(data_frame, label_matrix, val_idx,   val_tf,   path_lookup)

    # Class-balanced sampling
    class_counts = label_matrix[train_idx].sum(axis=0)
    class_weights = 1.0 / (class_counts + 1e-6) ** SAMPLER_POWER
    sample_weights = (label_matrix[train_idx] * class_weights).sum(axis=1)
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE_TRAIN,
        sampler=sampler,
        num_workers=1,
        persistent_workers=False,
        worker_init_fn=worker_init_fn,
        pin_memory=True,
    )

    val_loader  = DataLoader(val_ds,   BATCH_SIZE_VAL, shuffle=False,
                            num_workers=1, pin_memory=True,
                            worker_init_fn=worker_init_fn,
                            persistent_workers=False)
    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

    # Model
    
    
    base = timm.create_model(
        # "swinv2_base_window8_256",
        # "swinv2_small_window16_256",
        "swinv2_small_window8_256",
        img_size=256,
        pretrained=True,
        num_classes=0,
        # strict_img_size=False
    )
    # msg = base.load_state_dict(state_dict, strict=True)
    # print(base.embed_dim)
    # print(base.num_features)

    model = SwinWithView(base, NUM_CLASSES).to(device)


    # param_group = model.parameters()
    param_group = build_param_groups(model, base_lr=BASE_LR, decay=0.8)
    scaler = GradScaler(device="cuda")


    # Training setup
    criterion = AsymmetricLoss()

    optimizer = torch.optim.AdamW(param_group, weight_decay=WEIGHT_DECAY)
    # linear LR warmup for first 2 epochs, then cosine decay

    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=WARMUP_START_FACTOR,
        end_factor=WARMUP_END_FACTOR,
        total_iters=WARMUP_EPOCHS
    )

    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS - WARMUP_EPOCHS,
        eta_min=1e-6
    )

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_EPOCHS]
    )

    best_val   = 0.0
    no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        # update transform for progressive resizing
        train_ds.transform = make_train_tf(PROG_SCHEDULE[epoch])

        tr_loss, tr_auc, _ = run_epoch(train_loader, train=True, scaler=scaler)
        val_loss, val_auc, per_class = run_epoch(val_loader,   train=False, scaler=None)

        print("  Per-class AUCs:")
        for cls, auc in sorted(per_class.items(), key=lambda x: x[1]):
            print(f"    {cls:<20s} {auc:.3f}") 

        scheduler.step()

        flag = ""
        if val_auc > best_val:  # track best AUC
            best_val = val_auc
            no_improve = 0
            torch.save(model.state_dict(), MODEL_OUTPUT_FILE)
            flag = "  ->  saved"
        else:
            no_improve += 1
            flag = f"  (no improvement {no_improve}/{PATIENCE})"

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
            f"train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  "
            f"val_auc={val_auc:.4f}{flag}")

        if no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            break
    print("Done. Best val AUC:", round(best_val, 4))
    


Device: cuda
Found 112,120 images on disk.
Matched 112,120 rows after filtering.
No-Finding kept: 34,185 / 51,278  |  New train size: 78,209
Train: 78,209  |  Val: 16,818


train:   0%|          | 0/2445 [00:00<?, ?it/s]

In [ ]:
sd = extract_state_dict(ckpt)
print(type(sd))
print(len(sd))
print(list(sd.keys())[:20])


for k in list(sd.keys())[:50]:
    print(k, sd[k].shape)

<class 'collections.OrderedDict'>
476
['encoder.mask_token', 'encoder.patch_embed.proj.weight', 'encoder.patch_embed.proj.bias', 'encoder.patch_embed.norm.weight', 'encoder.patch_embed.norm.bias', 'encoder.layers.0.blocks.0.norm1.weight', 'encoder.layers.0.blocks.0.norm1.bias', 'encoder.layers.0.blocks.0.attn.logit_scale', 'encoder.layers.0.blocks.0.attn.q_bias', 'encoder.layers.0.blocks.0.attn.v_bias', 'encoder.layers.0.blocks.0.attn.relative_coords_table', 'encoder.layers.0.blocks.0.attn.relative_position_index', 'encoder.layers.0.blocks.0.attn.rpe_mlp.0.weight', 'encoder.layers.0.blocks.0.attn.rpe_mlp.0.bias', 'encoder.layers.0.blocks.0.attn.rpe_mlp.2.weight', 'encoder.layers.0.blocks.0.attn.qkv.weight', 'encoder.layers.0.blocks.0.attn.proj.weight', 'encoder.layers.0.blocks.0.attn.proj.bias', 'encoder.layers.0.blocks.0.norm2.weight', 'encoder.layers.0.blocks.0.norm2.bias']
encoder.mask_token torch.Size([1, 1, 96])
encoder.patch_embed.proj.weight torch.Size([96, 3, 4, 4])
encoder.pat

In [11]:
imgs, lbls, views = next(iter(train_loader))
out = model(imgs.to(device), views.to(device))